In [1]:
import torch.nn as nn
import torch

In [2]:
# 띄어쓰기 단위로 분리
input_text = "나는 최근 파리 여행을 다녀왔다"
input_text_list = input_text.split()
print("input_text_list: ", input_text_list)

# 토큰 -> 아이디 딕셔너리와 아이디 -> 토큰 딕셔너리 만들기
str2idx = {word: idx for idx, word in enumerate(input_text_list)}
idx2str = {idx: word for idx, word in enumerate(input_text_list)}
print("str2idx: ", str2idx)
print("idx2str: ", idx2str)

# 토큰을 토큰 아이디로 변환
input_ids = [str2idx[word] for word in input_text_list]
print("input_ids: ", input_ids)

input_text_list:  ['나는', '최근', '파리', '여행을', '다녀왔다']
str2idx:  {'나는': 0, '최근': 1, '파리': 2, '여행을': 3, '다녀왔다': 4}
idx2str:  {0: '나는', 1: '최근', 2: '파리', 3: '여행을', 4: '다녀왔다'}
input_ids:  [0, 1, 2, 3, 4]


In [3]:
embedding_dim = 16
max_position = 12
# 토큰 임베딩 층 생성
embed_layer = nn.Embedding(len(str2idx), embedding_dim)
# 위치 인코딩 층 생성
position_embed_layer = nn.Embedding(max_position, embedding_dim)

position_ids = torch.arange(len(input_ids), dtype=torch.long).unsqueeze(0)
position_encodings = position_embed_layer(position_ids)
token_embeddings = embed_layer(torch.tensor(input_ids))  # (5, 16)
token_embeddings = token_embeddings.unsqueeze(0)  # (1, 5, 16)
# 토큰 임베딩과 위치 인코딩을 더해 최종 입력 임베딩 생성
input_embeddings = token_embeddings + position_encodings
input_embeddings.shape

torch.Size([1, 5, 16])

In [4]:
head_dim = 16

weight_q = nn.Linear(embedding_dim, head_dim)
weight_k = nn.Linear(embedding_dim, head_dim)
weight_v = nn.Linear(embedding_dim, head_dim)

queries = weight_q(input_embeddings)
keys = weight_k(input_embeddings)
values = weight_v(input_embeddings)

In [5]:
from math import sqrt
import torch.nn.functional as F


# 내 이해: 정적인 워드 임베딩에서 주변 토큰의 정를 적용한 임베딩으로 변경
def compute_attention(queries, keys, values, is_causal=False):
    dim_k = queries.size(-1)  # 16
    # (1, 5, 16) matmul (1, 16, 5) -> (1, 5, 5) / 4
    scores = queries @ keys.transpose(-2, -1) / sqrt(dim_k)
    weights = F.softmax(scores, dim=-1)
    return weights @ values

In [6]:
print("원본 입력 형태:", input_embeddings.shape)

after_attention_embeddings = compute_attention(queries, keys, values)

print("어텐션 적용 후 형태:", after_attention_embeddings.shape)

원본 입력 형태: torch.Size([1, 5, 16])
어텐션 적용 후 형태: torch.Size([1, 5, 16])


In [9]:
class AttentionHead(nn.Module):
    def __init__(self, token_embed_dim, head_dim, is_causal=False):
        super().__init__()
        self.is_causal = is_causal
        self.weight_q = nn.Linear(token_embed_dim, head_dim)
        self.weight_k = nn.Linear(token_embed_dim, head_dim)
        self.weight_v = nn.Linear(token_embed_dim, head_dim)

    def forward(self, queries, keys, values):
        return compute_attention(
            self.weight_q(queries),
            self.weight_k(keys),
            self.weight_v(values),
            is_causal=self.is_causal
        )

In [10]:
attention_head = AttentionHead(embedding_dim, embedding_dim)
after_attention_embeddings = attention_head(input_embeddings, input_embeddings, input_embeddings)

In [16]:
class MultiHeadAttention(nn.Module):
    def __init__(self, token_embed_dim, d_model, n_head, is_causal=False):
        super().__init__()
        self.n_head = n_head
        self.is_causal = is_causal
        self.weight_q = nn.Linear(token_embed_dim, d_model)
        self.weight_k = nn.Linear(token_embed_dim, d_model)
        self.weight_v = nn.Linear(token_embed_dim, d_model)
        self.concat_linear = nn.Linear(d_model, d_model)
    
    def forward(self, queries, keys, values):
        B, T, C = queries.size()
        # [1, 5, 16] -> [1, 5, d_model] -> [1, 5, n_head, d_model/n_head] -> [1, n_head, 5, d_model/n_head]]
        queries = self.weight_q(queries).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        keys = self.weight_k(keys).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        values = self.weight_v(values).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        attention = compute_attention(queries, keys, values, is_causal=self.is_causal)
        output = attention.transpose(1, 2).contiguous().view(B, T, C)
        return self.concat_linear(output)


In [17]:
n_head = 4
multi_head_attention = MultiHeadAttention(embedding_dim, embedding_dim, n_head)
after_multi_head_attention_embeddings = multi_head_attention(input_embeddings, input_embeddings, input_embeddings)
print("멀티 헤드 어텐션 적용 후 형태:", after_multi_head_attention_embeddings.shape)

멀티 헤드 어텐션 적용 후 형태: torch.Size([1, 5, 16])


In [23]:
norm = nn.LayerNorm(embedding_dim)
norm_x = norm(input_embeddings)
print("층 정규화 결과:", norm_x.shape, ", ", norm_x.mean(dim=-1).data)

층 정규화 결과: torch.Size([1, 5, 16]) ,  tensor([[-3.3528e-08, -7.4506e-09,  3.7253e-08,  2.2352e-08, -2.2352e-08]])


In [24]:
class PreLayerNormFeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, src):
        x = self.norm(src)
        x = x + self.linear2(self.dropout1(self.activation(self.linear1(x))))
        x = self.dropout2(x)
        return x


In [25]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, d_model, nhead)
        self.norm1 = nn.LayerNorm(d_model)
        self.feedforward = PreLayerNormFeedForward(d_model, dim_feedforward, dropout)

    def forward(self, src):
        norm_x = self.norm1(src)
        attention_output = self.attention(norm_x, norm_x, norm_x)
        x = src + attention_output
        return self.feedforward(x)


In [ ]:
import copy

def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers):
        super().__init__()
        self.layers = get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src):
        output = src
        for layer in self.layers:
            output = layer(src)
        return output

In [31]:
def compute_atttention(queries, keys, values, is_causal=False):
    dim_k = queries.size(-1)  # 16
    scores = queries @ keys.transpose(-2, -1) / sqrt(dim_k)
    if is_causal:
        queries_length = queries.size(-2)
        key_length = keys.size(-2)
        temp_mask = torch.ones(queries_length, key_length, dtype=torch.bool).tril(diagonal=0)
        scores = scores.masked_fill(temp_mask == False, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    print("어텐션 가중치:", weights)
    return weights @ values

In [33]:
_ = compute_atttention(queries, keys, values, is_causal=True)

어텐션 가중치: tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.6663, 0.3337, 0.0000, 0.0000, 0.0000],
         [0.2342, 0.4424, 0.3234, 0.0000, 0.0000],
         [0.1162, 0.4537, 0.1457, 0.2844, 0.0000],
         [0.2649, 0.3464, 0.0949, 0.1257, 0.1681]]],
       grad_fn=<SoftmaxBackward0>)


In [34]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, d_model, nhead, is_causal=True)
        self.cross_attention = MultiHeadAttention(d_model, d_model, nhead)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.feedforward = PreLayerNormFeedForward(d_model, dim_feedforward, dropout)
    def forward(self, tgt, encoder_output, is_causal=True):
        x = self.norm1(tgt)
        x = x + self.dropout1(self.self_attention(x, x, x, is_causal=is_causal))
        x = self.norm2(x)
        x = x + self.dropout2(self.cross_attention(x, encoder_output, encoder_output))
        return self.feedforward(x)

In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers):
        super().__init__()
        self.layers = get_clones(decoder_layer, num_layers)
        self.num_layers = num_layers

    def forward(self, tgt, src):
        output = tgt
        for layer in self.layers:
            output = layer(output, src)
        return output